# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/susheel123-sketch/Flyrank-Internship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

**Lane:** Lane 2 — Refresh / Content Opportunity Scoring. **Model under audit:** my Week-5 notebook (`w05_model.ipynb`).
**Data:** the same starter slice (30,000 pages, 32 clients). **Label:** `trend_direction == "down"`. **Metric:** precision@K next to the base rate, plus AUC.
**Paper:** *The State of AI-Driven SEO, March 2026* (`docs/flyrank-seo-research-march-2026.pdf`). **Seed:** 42.

In [1]:
import os, sys, json, subprocess, warnings
import numpy as np, pandas as pd, sklearn
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
from sklearn.model_selection import GroupKFold, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, accuracy_score

IN_COLAB = "google.colab" in sys.modules
REPO_URL, REPO_DIR = "https://github.com/susheel123-sketch/Flyrank-Internship-ml", "Flyrank-Internship-ml"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")
SEED = 42
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
y = (df["trend_direction"] == "down").astype(int).values
groups = df["client_id"].values
print("rows:", len(df), "| clients:", df["client_id"].nunique(), f"| base rate: {y.mean():.3f}",
      f"| majority-class accuracy: {max(y.mean(), 1 - y.mean()):.3f}")
print("sklearn", sklearn.__version__)

rows: 30000 | clients: 32 | base rate: 0.542 | majority-class accuracy: 0.542
sklearn 1.9.1


## 1. Two paper findings + my methodology questions

The paper is upfront about its standards: it says headline findings are **observational**, ML pages are an **exploratory appendix**,
no p-values are reported, and age confounds some comparisons. These questions are the ones I'd want asked of my own work — they start from that disclosure.

### Finding A — Finding #4, "The Freshness Multiplier" (and Playbook action #1)
**The claim:** 365+ day content refreshed within the last 30 days shows **3.2× health and 57× impressions** (71 → 4,039) vs unrefreshed old content, and "refresh timing is one of the strongest measured levers".

- **Where does the comparison come from?** Two groups at one moment: old pages updated recently vs old pages not updated. "Refreshed" is a team's *decision*, not a random event.
- **Question 1 — which way does the arrow run?** The playbook itself says to refresh pages "with proven historical visibility". If teams pick pages that already earn impressions, the refreshed group starts ahead. A 57× gap could be partly *who got picked*, not *what the refresh did*.
- **Question 2 — is it a page effect or a client effect?** Refreshes often happen in client-wide batches. If one client refreshes everything and another refreshes nothing, the comparison is mostly between clients.
- **Question 3 — survivor filter:** the extended cuts use the active subset (impressions > 0 and sessions > 0). The unrefreshed 365+ group (avg 71 impressions) may be mostly near-dead pages that barely passed the filter.
- **What would carry the claim:** the paper's own "Measure" line already points the right way — *refreshed vs comparable untouched pages*. The concrete version: the same pages' impressions in the 30 days **before vs after** their refresh date, against untouched pages of the same client and age band.
- **My slice says this matters (cell below):** recently-updated pages here have **lower** traffic, not higher. 15 of 32 clients sit almost entirely on one side of the 30-day line, and inside mixed clients the recently-updated side is lower in 11 of 14. In my data a pooled freshness gap is mostly a client-batch pattern. I can't test the paper's data, but this is why the within-client question is worth asking.

### Finding B — ML appendix, "What Predicts Growth?"
**The claim:** logistic regression with **71% holdout accuracy** (80/20 split) separates growing from declining pages; content age is the strongest negative signal, while days visible and recent impressions are strong positive signals.

- **Where does the label come from?** Trend direction = last-30-day vs previous-30-day impressions. That is the same family of label as mine.
- **Question 1 — what is 71% measured against?** From the paper's own counts (74,187 rising vs 45,272 falling), always guessing "up" scores about **62%**. So the model adds roughly **9 points** — useful to state next to the 71%.
- **Question 2 — do features and label share a window?** "Recent impressions" and "days visible" are 90-day measures that **contain** the 60 days the label is built from. That is a same-window link: it describes, it doesn't forecast.
- **Question 3 — does the split support "predicts"?** An 80/20 split is usually random by page, so pages from the same brand sit on both sides. Section 2 shows that on my slice this alone adds **+0.10 AUC** for boosting.
- **What would carry the claim:** report the majority-class baseline, split by brand (or by time), and build features only from before the label window. The paper already labels this "descriptive" and "exploratory" — these steps would let it say more.

In [2]:
# Two small checks that make the questions above concrete (my slice, not the paper's data).

# (a) The paper's own counts for its growth model (Finding #1 table: 74,187 rising vs 45,272 falling).
rising, falling = 74_187, 45_272
print(f"(a) Majority-class accuracy implied by the paper's up/down counts: {rising / (rising + falling):.3f} "
      f"-> a 71% holdout accuracy is about {0.71 - rising / (rising + falling):+.3f} above always guessing 'up'")

# (b) Freshness in MY slice: pooled vs inside-client. If refreshes happen in client batches,
#     a pooled 'refreshed vs not' gap can be a between-client gap.
t = df.assign(y=y, refreshed_30d=np.where(df["days_since_last_update"] <= 30, "updated <=30d", "updated >30d"))
pooled = t.groupby("refreshed_30d").agg(n=("y", "size"), median_impressions=("impressions_90d", "median"),
                                          decline_rate=("y", "mean")).round(3)
print("(b1) Pooled comparison"); display(pooled)
mix = t.groupby("client_id")["refreshed_30d"].agg(lambda s: (s == "updated <=30d").mean())
print(f"(b2) Clients where >=95% of pages fall on ONE side of the 30-day line: {((mix >= 0.95) | (mix <= 0.05)).sum()} of {len(mix)}")
both = t.groupby(["client_id", "refreshed_30d"])["impressions_90d"].agg(["size", "median"]).unstack().dropna()
both = both[(both[("size", "updated <=30d")] >= 30) & (both[("size", "updated >30d")] >= 30)]
ratio = both[("median", "updated <=30d")] / both[("median", "updated >30d")].replace(0, np.nan)
print(f"(b3) Inside the {len(both)} clients that have >=30 pages on both sides: median impressions ratio "
      f"(recently updated / not) = {ratio.median():.2f}; recently-updated side higher in {(ratio > 1).sum()} of {len(both)}")

(a) Majority-class accuracy implied by the paper's up/down counts: 0.621 -> a 71% holdout accuracy is about +0.089 above always guessing 'up'
(b1) Pooled comparison


,n,median_impressions,decline_rate
refreshed_30d,,,
updated <=30d,20480,470.0,0.511
updated >30d,9520,1576.0,0.608


(b2) Clients where >=95% of pages fall on ONE side of the 30-day line: 15 of 32
(b3) Inside the 14 clients that have >=30 pages on both sides: median impressions ratio (recently updated / not) = 0.67; recently-updated side higher in 3 of 14


## 2. My model under an honest split (before/after)

**Improvement 1 — random split → grouped by client.** My Week-5 notebook already used the grouped split. Here I show what I *would* have reported with a random split, so the gap is visible.

In [3]:
LOG = ["impressions_90d", "clicks_90d", "sessions_90d", "engaged_sessions_90d", "ai_sessions_90d", "search_volume", "cpc"]
NUM = LOG + ["competition", "word_count", "days_with_impressions", "days_with_sessions", "content_age_days",
             "days_since_last_update", "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct"]
CAT = ["competition_level", "content_type", "main_intent"]

def week5_features(d):
    # Exactly the Week-5 feature builder.
    X = pd.DataFrame(index=d.index)
    for c in NUM:
        X[c] = (np.log1p(d[c]) if c in LOG else d[c]).fillna(0)
        if d[c].isna().any():
            X["has_" + c] = d[c].notna().astype(int)
    X["has_position"] = (d["avg_position"] > 0).astype(int)
    return X.join(pd.get_dummies(d[CAT].fillna("unknown"), dtype=int))

MODELS = {"Logistic Regression": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
          "Gradient Boosting": lambda: HistGradientBoostingClassifier(random_state=SEED)}
RANDOM_SPLIT = list(KFold(5, shuffle=True, random_state=SEED).split(df))
GROUPED_SPLIT = list(GroupKFold(5).split(df, y, groups))

def p_at_k(y_true, s, k):
    return float(np.asarray(y_true)[np.argsort(-np.asarray(s), kind="stable")[:k]].mean())

def evaluate(X, folds, make):
    oof, rows = np.zeros(len(y)), []
    for tr, te in folds:
        p = make().fit(X.iloc[tr], y[tr]).predict_proba(X.iloc[te])[:, 1]; oof[te] = p
        rows.append([p_at_k(y[te], p, 50), roc_auc_score(y[te], p), accuracy_score(y[te], p > 0.5)])
    m = np.mean(rows, axis=0)
    return {"per-fold P@50": m[0], "AUC": m[1], "accuracy": m[2],
            "pooled P@50": p_at_k(y, oof, 50), "pooled P@100": p_at_k(y, oof, 100)}, oof

X5 = week5_features(df)
rows = []
for name, make in MODELS.items():
    for split_name, folds in [("BEFORE: random 5-fold", RANDOM_SPLIT), ("AFTER: grouped by client", GROUPED_SPLIT)]:
        r, _ = evaluate(X5, folds, make)
        rows.append({"model": name, "split": split_name, **{k: round(v, 3) for k, v in r.items()}})
split_df = pd.DataFrame(rows)
display(split_df)
for name in MODELS:
    a, b = split_df.query("model==@name")["AUC"].tolist()
    print(f"{name}: AUC drops {a - b:.3f} when the model can no longer see the test pages' client.")

,model,split,per-fold P@50,AUC,accuracy,pooled P@50,pooled P@100
0,Logistic Regression,BEFORE: random 5-fold,0.868,0.716,0.661,0.92,0.87
1,Logistic Regression,AFTER: grouped by client,0.776,0.659,0.620,0.86,0.81
2,Gradient Boosting,BEFORE: random 5-fold,0.972,0.781,0.705,0.96,0.94
3,Gradient Boosting,AFTER: grouped by client,0.832,0.685,0.649,0.84,0.84


Logistic Regression: AUC drops 0.057 when the model can no longer see the test pages' client.
Gradient Boosting: AUC drops 0.096 when the model can no longer see the test pages' client.


**Reading improvement 1:**
- With a random split, boosting looks excellent: AUC 0.78, pooled P@50 0.96. With a grouped split it drops to 0.69 / 0.84.
- That gap (−0.10 AUC) is how much the random split was letting the model **memorise clients**. Logistic Regression drops less (−0.06) because it has less room to memorise.
- The random-split accuracy (0.70) is close to the paper's 71% — a reminder of how far a random split can flatter this kind of label.

**Improvement 2 — a time-aware feature window.** Grouping fixes *who* the model sees. It does not fix *when*. Most Week-5 features cover days 1–90, and the label lives in days 1–60. Below, I rebuild a feature set that uses only what was knowable at day 60.

In [4]:
# Second improvement — a time-aware FEATURE window.
# The label compares days 1-30 with days 31-60 (counting back from export). The 90-day totals cover days 1-90,
# so they include the label period. Days 61-90 can be rebuilt exactly: 90d total - last30 - prev30.
w61 = pd.DataFrame({"impr": df["impressions_90d"] - df["impressions_last_30d"] - df["impressions_prev_30d"],
                    "clicks": df["clicks_90d"] - df["clicks_last_30d"] - df["clicks_prev_30d"],
                    "sess": df["sessions_90d"] - df["sessions_last_30d"] - df["sessions_prev_30d"]})
assert (w61 >= 0).all().all(), "61-90 window went negative — identity broken"
print("Days 61-90 rebuilt; all values >= 0 (asserted). Median impressions in that window:", w61["impr"].median())

def strict_features(d):
    # Only things knowable on the day the label window starts (60 days before export).
    X = pd.DataFrame(index=d.index)
    for c in ["search_volume", "cpc"]:
        X[c] = np.log1p(d[c]).fillna(0); X["has_" + c] = d[c].notna().astype(int)
    X["competition"] = d["competition"].fillna(0)
    X["word_count"] = d["word_count"].fillna(0); X["has_word_count"] = d["word_count"].notna().astype(int)
    X["age_at_cutoff"] = d["content_age_days"] - 60
    X["impr_61_90"] = np.log1p(w61["impr"]); X["clicks_61_90"] = np.log1p(w61["clicks"]); X["sess_61_90"] = np.log1p(w61["sess"])
    X["ctr_61_90"] = np.where(w61["impr"] > 0, w61["clicks"] / w61["impr"].replace(0, 1) * 100, 0)
    X["visible_61_90"] = (w61["impr"] > 0).astype(int)
    return X.join(pd.get_dummies(d[CAT].fillna("unknown"), dtype=int))

XS = strict_features(df)
OOF = {}
rows = []
for name, make in MODELS.items():
    for feat_name, XX in [("Week-5 features (overlap label window)", X5), ("strict timeline (days 61-90 + static)", XS)]:
        r, oof = evaluate(XX, GROUPED_SPLIT, make); OOF[(name, feat_name)] = oof
        rows.append({"model": name, "features": feat_name, **{k: round(v, 3) for k, v in r.items()}})
time_df = pd.DataFrame(rows); display(time_df)

# Is the boosting drop about the time window, or mostly about losing avg_position (a 90-day average)?
XSP = XS.assign(avg_position=df["avg_position"], has_position=(df["avg_position"] > 0).astype(int))
for name, make in MODELS.items():
    r, _ = evaluate(XSP, GROUPED_SPLIT, make)
    print(f"{name}, strict timeline + avg_position added back: AUC {r['AUC']:.3f}, pooled P@50 {r['pooled P@50']:.2f}")

Days 61-90 rebuilt; all values >= 0 (asserted). Median impressions in that window: 308.0


,model,features,per-fold P@50,AUC,accuracy,pooled P@50,pooled P@100
0,Logistic Regression,Week-5 features (overlap label window),0.776,0.659,0.620,0.86,0.81
1,Logistic Regression,strict timeline (days 61-90 + static),0.816,0.663,0.635,0.86,0.85
2,Gradient Boosting,Week-5 features (overlap label window),0.832,0.685,0.649,0.84,0.84
3,Gradient Boosting,strict timeline (days 61-90 + static),0.680,0.612,0.611,0.60,0.59


Logistic Regression, strict timeline + avg_position added back: AUC 0.664, pooled P@50 0.84


Gradient Boosting, strict timeline + avg_position added back: AUC 0.643, pooled P@50 0.66


In [5]:
# The Week-4 baseline on the same grouped split, as-is and on the strict timeline (CTR + impressions from days 61-90;
# position is only available as a 90-day average, so that one overlap remains and is disclosed).
def baseline(train, test, impr, ctr):
    band = lambda d: pd.cut(d["avg_position"].where(d["avg_position"] > 0), [0, 3, 10, 20], labels=["1-3", "4-10", "11-20"])
    btr = band(train); ok = (impr[train.index] >= 500) & btr.notna()
    med = ctr[train.index][ok].groupby(btr[ok], observed=True).median()
    bte = band(test); e = bte.map(med).astype(float)
    it, ct = impr[test.index], ctr[test.index]
    flag = (it >= 500) & bte.notna() & (ct < 0.5 * e)
    return np.where(flag, (1 - ct / e).clip(0, 1) * np.log1p(it), 0.0) + np.log1p(it.values) * 1e-6

ctr61 = pd.Series(np.where(w61["impr"] > 0, w61["clicks"] / w61["impr"].replace(0, 1) * 100, 0), index=df.index)
B = {}
for label, impr, ctr in [("Week-4 baseline (as built)", df["impressions_90d"], df["ctr"]),
                         ("Week-4 baseline (strict timeline)", w61["impr"] * 3, ctr61)]:   # x3 keeps the 500/90d floor comparable
    s = np.zeros(len(df))
    for tr, te in GROUPED_SPLIT:
        s[te] = baseline(df.iloc[tr], df.iloc[te], impr, ctr)
    B[label] = s

final_rows = []
for label, s in B.items():
    final_rows.append({"ranker": label, **{f"pooled P@{k}": round(p_at_k(y, s, k), 3) for k in [20, 50, 100, 200]}, "AUC (pooled)": np.nan})
for (name, feat), s in OOF.items():
    final_rows.append({"ranker": f"{name} — {feat}", **{f"pooled P@{k}": round(p_at_k(y, s, k), 3) for k in [20, 50, 100, 200]},
                       "AUC (pooled)": round(roc_auc_score(y, s), 3)})
before_after = pd.DataFrame(final_rows).set_index("ranker"); before_after["base rate"] = round(y.mean(), 3)
print("All rankers, grouped-by-client split, out-of-fold, one pooled queue:"); display(before_after)

All rankers, grouped-by-client split, out-of-fold, one pooled queue:


,pooled P@20,pooled P@50,pooled P@100,pooled P@200,AUC (pooled),base rate
ranker,,,,,,
Week-4 baseline (as built),0.65,0.60,0.66,0.685,NaN,0.542
Week-4 baseline (strict timeline),0.65,0.70,0.75,0.720,NaN,0.542
Logistic Regression — Week-5 features (overlap label window),0.80,0.86,0.81,0.820,0.654,0.542
Logistic Regression — strict timeline (days 61-90 + static),0.90,0.86,0.85,0.850,0.661,0.542
Gradient Boosting — Week-5 features (overlap label window),0.90,0.84,0.84,0.845,0.693,0.542
Gradient Boosting — strict timeline (days 61-90 + static),0.75,0.60,0.59,0.620,0.632,0.542


**Reading improvement 2 (the most important result in this notebook):**
- **Logistic Regression holds up:** strict-timeline pooled P@50 0.86 (same as before), AUC 0.66. Its signal does not depend on seeing the label window.
- **Gradient Boosting does not:** strict-timeline pooled P@50 **0.60** — no better than the Week-4 rule. Adding `avg_position` back only lifts it to 0.66.
  So most of boosting's Week-5 edge came from features that **overlap the label window**.
- **The Week-4 rule also improves on the strict timeline** (P@50 0.60 → 0.70): CTR measured *before* the label window is a cleaner signal than CTR that includes it.
- **Decision:** my Week-5 pick changes from Gradient Boosting to **Logistic Regression on strict-timeline features**. It is simpler, and it is the one whose numbers survive both fixes.

## 3. Leakage audit

The same hunt as Week 3, on the final feature set: timeline first, then attack each suspect.

In [6]:
# 3a. Timeline: where each feature's data comes from, relative to the label window (days 1-60 back from export).
timeline = pd.DataFrame([
    ("trend_direction / trend_pct", "days 1-60", "IS the label", "excluded"),
    ("impressions/clicks/sessions_last_30d, _prev_30d", "days 1-60", "builds the label", "excluded"),
    ("impressions_90d, clicks_90d, sessions_90d, ctr", "days 1-90", "contains the label window", "Week 5 used -> strict set replaces with days 61-90"),
    ("days_with_impressions / days_with_sessions", "days 1-90", "contains the label window", "Week 5 used -> dropped in strict set"),
    ("avg_position, engagement_rate, scroll_rate, ai_traffic_pct", "days 1-90 (average)", "contains the label window", "Week 5 used -> dropped in strict set"),
    ("days_since_last_update", "measured at export", "an update INSIDE the label window is a future event", "Week 5 used -> dropped in strict set"),
    ("content_age_days", "measured at export", "shift by 60 days = age at cutoff", "kept (shifted)"),
    ("word_count, search_volume, cpc, competition, content_type, main_intent", "page metadata", "no time window", "kept (caveat: word_count could change on update)"),
], columns=["feature", "window", "relation to label", "decision"])
display(timeline)
upd = df["days_since_last_update"]
print(f"Pages updated INSIDE the label window (<=60 days before export): {(upd <= 60).sum()} of {len(df)} ({(upd <= 60).mean():.0%})")

,feature,window,relation to label,decision
0,trend_direction / trend_pct,days 1-60,IS the label,excluded
1,"impressions/clicks/sessions_last_30d, _prev_30d",days 1-60,builds the label,excluded
2,"impressions_90d, clicks_90d, sessions_90d, ctr",days 1-90,contains the label window,Week 5 used -> strict set replaces with days 6...
3,days_with_impressions / days_with_sessions,days 1-90,contains the label window,Week 5 used -> dropped in strict set
4,"avg_position, engagement_rate, scroll_rate, ai...",days 1-90 (average),contains the label window,Week 5 used -> dropped in strict set
5,days_since_last_update,measured at export,an update INSIDE the label window is a future ...,Week 5 used -> dropped in strict set
6,content_age_days,measured at export,shift by 60 days = age at cutoff,kept (shifted)
7,"word_count, search_volume, cpc, competition, c...",page metadata,no time window,kept (caveat: word_count could change on update)


Pages updated INSIDE the label window (<=60 days before export): 20608 of 30000 (69%)


In [7]:
# 3b. Harness test: ADD the label source and the score must jump to ~1.0. If it doesn't, the test bench is broken.
leak = X5.assign(trend_pct=df["trend_pct"].fillna(0))
r_leak, _ = evaluate(leak, GROUPED_SPLIT, MODELS["Gradient Boosting"])
print(f"WITH trend_pct (planted leak): AUC {r_leak['AUC']:.3f}, pooled P@50 {r_leak['pooled P@50']:.2f}  <- confession, harness works")

# 3c. Train-without test on each overlapping group (Gradient Boosting, grouped split). Big drops = the model leaned on them.
suspects = {"90-day traffic totals + ctr": ["impressions_90d", "clicks_90d", "sessions_90d", "engaged_sessions_90d", "ai_sessions_90d", "ctr"],
            "window-shape (days with traffic)": ["days_with_impressions", "days_with_sessions"],
            "days_since_last_update": ["days_since_last_update"],
            "90-day averages (position, rates)": ["avg_position", "has_position", "engagement_rate", "scroll_rate", "has_scroll_rate", "ai_traffic_pct"]}
full, _ = evaluate(X5, GROUPED_SPLIT, MODELS["Gradient Boosting"])
rows = [{"removed": "nothing (Week-5 model)", "AUC": round(full["AUC"], 3), "pooled P@50": round(full["pooled P@50"], 3)}]
for label, cols in suspects.items():
    r, _ = evaluate(X5.drop(columns=cols), GROUPED_SPLIT, MODELS["Gradient Boosting"])
    rows.append({"removed": label, "AUC": round(r["AUC"], 3), "pooled P@50": round(r["pooled P@50"], 3)})
display(pd.DataFrame(rows))

# 3d. 'Too good to be true' scan: best single-feature AUC (either direction) on the Week-5 set.
single = {c: max(roc_auc_score(y, X5[c]), 1 - roc_auc_score(y, X5[c])) for c in X5.columns if X5[c].nunique() > 1}
print("Strongest single features (AUC, either direction):", {k: round(v, 3) for k, v in sorted(single.items(), key=lambda kv: -kv[1])[:4]})

# 3e. Product flags + population.
flag_like = [c for c in df.columns if any(s in c.lower() for s in ["flag", "health", "eligible", "optimiz", "score"])]
print("Product-flag-like columns in the data:", flag_like or "none")
print(f"Population filters baked into the slice: impressions_90d >= 1 for every row ({(df['impressions_90d'] >= 1).all()}), "
      f"content_age_days >= 90 ({(df['content_age_days'] >= 90).all()})")

WITH trend_pct (planted leak): AUC 1.000, pooled P@50 1.00  <- confession, harness works


,removed,AUC,pooled P@50
0,nothing (Week-5 model),0.685,0.84
1,90-day traffic totals + ctr,0.666,0.82
2,window-shape (days with traffic),0.659,0.82
3,days_since_last_update,0.679,0.82
4,"90-day averages (position, rates)",0.658,0.76


Strongest single features (AUC, either direction): {'content_age_days': 0.591, 'impressions_90d': 0.584, 'days_with_impressions': 0.579, 'word_count': 0.565}
Product-flag-like columns in the data: none
Population filters baked into the slice: impressions_90d >= 1 for every row (True), content_age_days >= 90 (True)


**What the audit found:**
- **No direct label leak.** The label sources are excluded. The planted-leak test proves the harness would catch one (AUC jumps to 1.00).
  No single feature is suspiciously strong (best single-feature AUC is 0.59).
- **The real issue is window overlap, not one bad column.** Removing any one overlapping group costs boosting only 0.01–0.03 AUC — the overlap is spread across many features.
  That is why only the strict-timeline rebuild exposes it (section 2).
- **`days_since_last_update` was a quiet problem.** 69% of pages were updated **inside** the label window. An update in the middle of the window can *cause* the change the label measures — that is a future event, not a feature. Dropped in the strict set.
- **No product flags** in this slice (no health score, eligibility or optimisation-flag columns).
- **Population, disclosed:** every row has ≥1 impression in the 90 days and is ≥90 days old. "Has impressions in the window" uses outcome-window information — a choice, not a crime. It means results only apply to pages that were still showing up.

In [8]:
# 3f. Real failure examples from the most honest ranker (Logistic Regression, strict timeline, grouped split).
s = OOF[("Logistic Regression", "strict timeline (days 61-90 + static)")]
ex = df[["content_id", "content_type", "main_intent", "content_age_days", "days_since_last_update", "avg_position"]].copy()
ex["impr_61_90"] = w61["impr"]; ex["clicks_61_90"] = w61["clicks"]
ex["impr_prev_30d"] = df["impressions_prev_30d"]; ex["impr_last_30d"] = df["impressions_last_30d"]   # shown to EXPLAIN misses, not used by the model
ex["score"] = s.round(3); ex["declined"] = y
fp = ex.sort_values("score", ascending=False).query("declined == 0").head(3)
fn = ex.sort_values("score").query("declined == 1").head(3)
print("Ranked near the top, did NOT decline (false alarms):"); display(fp)
print("Ranked near the bottom, DID decline (misses):"); display(fn)
top = ex.nlargest(50, "score")
print(f"Top 50 of this ranker: {top['declined'].sum()} declined | zero clicks in days 61-90: {(top['clicks_61_90'] == 0).mean():.0%} | "
      f"updated inside the label window: {(top['days_since_last_update'] <= 60).mean():.0%} (vs {(df['days_since_last_update'] <= 60).mean():.0%} overall)")
# Label noise: a 'decline' can be 3 impressions -> 0. How many down-labels rest on tiny counts?
tiny = df["impressions_prev_30d"] < 20
print(f"Down-labels resting on <20 impressions in the earlier 30 days: {(tiny & (y == 1)).sum()} of {y.sum()} "
      f"({(tiny & (y == 1)).sum() / y.sum():.0%}); decline rate among those tiny pages: {y[tiny].mean():.2f}")

Ranked near the top, did NOT decline (false alarms):


,content_id,content_type,main_intent,content_age_days,days_since_last_update,avg_position,impr_61_90,clicks_61_90,impr_prev_30d,impr_last_30d,score,declined
27036,content_b2ec40b9d68c,keyword article,informational,139,104,47.3,5654,0,6947,10856,0.931,0
11868,content_013a5d3dfc26,keyword article,informational,148,104,25.7,20175,1,21793,23684,0.916,0
23793,content_873decd76dc4,keyword article,informational,118,20,49.2,2568,0,2690,2457,0.913,0


Ranked near the bottom, DID decline (misses):


,content_id,content_type,main_intent,content_age_days,days_since_last_update,avg_position,impr_61_90,clicks_61_90,impr_prev_30d,impr_last_30d,score,declined
18171,content_24796d98b025,keyword article,navigational,347,20,2.0,0,0,1,0,0.014,1
16382,content_3c9ecbee7605,keyword article,navigational,347,20,14.8,0,0,3,1,0.017,1
18494,content_13a801b1e212,keyword article,navigational,445,104,11.7,1050,6,287,213,0.054,1


Top 50 of this ranker: 42 declined | zero clicks in days 61-90: 78% | updated inside the label window: 38% (vs 69% overall)
Down-labels resting on <20 impressions in the earlier 30 days: 2445 of 16262 (15%); decline rate among those tiny pages: 0.33


**Real failure examples (strict-timeline Logistic Regression):**
- **False alarms are low-click, deep-ranked pages that were actually rising.** For example, `content_b2ec40b9d68c` sits at position 47, has zero clicks, and went from 6,947 to 10,856 impressions.
  The model reads "visible but nobody clicks" as decline risk. Some of those pages are just slowly climbing.
- **Misses are tiny pages whose "decline" is noise.** `content_24796d98b025` went from **1 impression to 0** — that counts as a 100% decline. 15% of all down-labels rest on fewer than 20 earlier impressions.
  No ranker should be blamed for missing those. A volume floor on the label would make the target cleaner.
- **One-pattern concentration again.** 78% of this ranker's top 50 had zero clicks in days 61–90. If zero clicks are sometimes a tracking gap (my Week-4 review suspected this), the queue would over-send reviewers to fine pages. Check this on the warehouse before using the queue.

## 4. Claim rewrite

The claims below are mine, from earlier notebooks. Each one went further than the evidence. The ledger cell under this section holds the numbers each rewrite rests on.

**1. Week 1 (w01):** *"A random forest model beat the hand-written baseline rule by roughly 3x … this tells me there's real, learnable signal in this data."*
→ **Rewrite:** "On a random split of the starter slice, a random forest ranked declining pages better than the reference rule. Under a client-grouped split with features from before the label window, a simpler model still shows a **measured, directional** edge over the rule (pooled P@50 0.86 vs 0.70, base rate 0.54)."

**2. Week 5 (w05):** *"Gradient Boosting beats my Week-4 rule … pooled P@50 0.84 vs 0.60."*
→ **Rewrite:** "Gradient Boosting's **observed** edge over the rule (0.84 vs 0.60) depended on features that overlap the label window. With those removed it fell to 0.60. The model whose result held under both checks is Logistic Regression (pooled P@50 0.86)."

**3. Week 2 (w02):** *the true target is "will this page's traffic meaningfully decline in the near future".*
→ **Rewrite:** "The model ranks pages by how much they resemble pages whose impressions fell between two recent 30-day windows. This is **decision-support** for review order, measured on one snapshot. It is **not** a forecast of future decline, and 15% of its 'declines' rest on fewer than 20 impressions."

**4. Week 4 (w04):** *the CTR-vs-position signal was "CONFIRMED".*
→ **Rewrite:** "Low CTR for a page's position band is **observed** alongside higher decline rates, inside 15 of 16 clients and on the same 90-day snapshot as the label. Measured with CTR from *before* the label window, the rule still ranks above the base rate (P@50 0.70). That is directional support, not proof that fixing CTR prevents decline." 

In [9]:
# The numbers each rewritten claim rests on, pulled from this run (so the text can't drift from the evidence).
g = lambda q, col: float(split_df.query(q)[col].iloc[0])
ledger = pd.DataFrame([
    ("Week-5 GB, random split", g("model=='Gradient Boosting' and split.str.startswith('BEFORE')", "AUC"), g("model=='Gradient Boosting' and split.str.startswith('BEFORE')", "pooled P@50")),
    ("Week-5 GB, grouped split", g("model=='Gradient Boosting' and split.str.startswith('AFTER')", "AUC"), g("model=='Gradient Boosting' and split.str.startswith('AFTER')", "pooled P@50")),
    ("GB, grouped + strict timeline", float(time_df.iloc[3]["AUC"]), float(time_df.iloc[3]["pooled P@50"])),
    ("LR, grouped + strict timeline", float(time_df.iloc[1]["AUC"]), float(time_df.iloc[1]["pooled P@50"])),
    ("Week-4 baseline, grouped + strict timeline", np.nan, float(before_after.loc["Week-4 baseline (strict timeline)", "pooled P@50"])),
], columns=["setting", "AUC", "pooled P@50"]).round(3)
ledger["base rate"] = round(y.mean(), 3)
display(ledger)
audit = {"assignment": "ML-09", "seed": SEED, "base_rate": round(float(y.mean()), 4),
         "split_before_after": split_df.to_dict(orient="records"), "timeline_before_after": time_df.to_dict(orient="records"),
         "pooled_rankers": before_after.reset_index().to_dict(orient="records"),
         "planted_leak_auc": round(r_leak["AUC"], 4), "claim_ledger": ledger.to_dict(orient="records")}
os.makedirs("work/outputs", exist_ok=True)
json.dump(audit, open("work/outputs/w06_audit_metrics.json", "w"), indent=2, default=float)
print("wrote work/outputs/w06_audit_metrics.json")

,setting,AUC,pooled P@50,base rate
0,"Week-5 GB, random split",0.781,0.96,0.542
1,"Week-5 GB, grouped split",0.685,0.84,0.542
2,"GB, grouped + strict timeline",0.612,0.60,0.542
3,"LR, grouped + strict timeline",0.663,0.86,0.542
4,"Week-4 baseline, grouped + strict timeline",NaN,0.70,0.542


wrote work/outputs/w06_audit_metrics.json


## What I'm carrying into the capstone

- **Model:** Logistic Regression on strict-timeline features, evaluated with a client-grouped split. Gradient Boosting's Week-5 lead does not survive the time-aware check.
- **Bar to beat:** the Week-4 rule on the strict timeline (pooled P@50 0.70), not the 0.60 version.
- **Open risks, stated plainly:**
  - Zero-click concentration at the top of the queue.
  - Label noise from tiny pages.
  - Population limited to pages still visible in the window.
  - Everything is one snapshot. The forward-window test needs the warehouse.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (executed before commit; seed 42)
- [x] No client names, URLs, or private queries anywhere (pseudonymous IDs only)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Two paper findings, each with a constructive methodology question
- [x] Before/after under an honest split (random → grouped) AND a time-aware feature window
- [x] Leakage audit: timeline, planted-leak harness test, train-without tests, flag + population check, real failure examples
- [x] Receipts committed: `work/outputs/w06_audit_metrics.json`